# GNN Retrieval Training from Neo4j (`Chunk`-`Entity`)

Notebook строит обучающий пайплайн для retrieval на графе Neo4j:
- читает структуру `(:Chunk)-[:MENTIONS]->(:Entity)` и `(:Entity)-[:RELATED_TO]->(:Entity)`,
- формирует гетерогенный граф,
- обучает GraphSAGE для ранжирования `Entity` по `Chunk` (link prediction),
- считает `Recall@K`/`MRR`,
- записывает `gnn_embedding` обратно в Neo4j.


In [2]:
# При необходимости раскомментируйте установку зависимостей
%pip install neo4j pandas numpy scikit-learn torch torch-geometric qdrant-client



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from neo4j import GraphDatabase
from qdrant_client import QdrantClient
from qdrant_client.models import FieldCondition, Filter, MatchAny
from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE


device(type='cuda')

In [32]:
# Конфиг подключения
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7688')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')

QDRANT_URL = os.getenv('QDRANT_URL', 'http://localhost:6333')
QDRANT_API_KEY = os.getenv('QDRANT_API_KEY')
QDRANT_CHUNK_COLLECTION = os.getenv('QDRANT_CHUNK_COLLECTION', 'legal_chunks_vectors')
QDRANT_ENTITY_COLLECTION = os.getenv('QDRANT_ENTITY_COLLECTION', 'legal_entities_vectors_test')
QDRANT_ID_FIELD = os.getenv('QDRANT_ID_FIELD', 'id')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

def run_query(query: str, params: dict | None = None):
    with driver.session() as session:
        result = session.run(query, params or {})
        return [r.data() for r in result]

schema_probe = run_query("""
MATCH (c:Chunk)
OPTIONAL MATCH (c)-[:MENTIONS]->(e:Entity)
RETURN count(DISTINCT c) AS chunks, count(DISTINCT e) AS entities, count(*) AS mention_rows
""")
schema_probe


[{'chunks': 837, 'entities': 9359, 'mention_rows': 13943}]

In [33]:
# Вытягиваем узлы/ребра и embedding_id для загрузки векторов из Qdrant
chunks = run_query("""
MATCH (c:Chunk)
RETURN
  c.chunk_id AS chunk_id,
  c.doc_id AS doc_id,
  c.embedding_id AS embedding_id,
  coalesce(c.text, '') AS text
""")

entities = run_query("""
MATCH (e:Entity)
RETURN
  e.entity_id AS entity_id,
  coalesce(e.embedding_id, e.entity_id) AS embedding_id,
  coalesce(e.entity_name, e.normalized_value, e.value, '') AS entity_text
""")

mentions = run_query("""
MATCH (c:Chunk)-[:MENTIONS]->(e:Entity)
RETURN c.chunk_id AS chunk_id, e.entity_id AS entity_id
""")

related = run_query("""
MATCH (e1:Entity)-[:RELATED_TO]->(e2:Entity)
RETURN e1.entity_id AS src_entity_id, e2.entity_id AS dst_entity_id
""")

chunks_df = pd.DataFrame(chunks).dropna(subset=['chunk_id', 'embedding_id']).drop_duplicates('chunk_id')
entities_df = pd.DataFrame(entities).dropna(subset=['entity_id', 'embedding_id']).drop_duplicates('entity_id')
mentions_df = pd.DataFrame(mentions).dropna().drop_duplicates()
related_df = pd.DataFrame(related).dropna().drop_duplicates()

print('chunks (with embedding_id):', len(chunks_df))
print('entities (with embedding_id):', len(entities_df))
print('mentions:', len(mentions_df))
print('related_to:', len(related_df))
chunks_df.head(2), entities_df.head(2), mentions_df.head(2)


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `text` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=7, column=14, offset=119>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 7, 'column': 14}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (c:Chunk)\nRETURN\n  c.chunk_id AS chunk_id,\n  c.doc_id AS doc_id,\n  c.embedding_id AS embedding_id,\n  coalesce(c.text, '') AS text\n"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `normalized_value` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=6, column=29, offset=138>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 138, 'line': 6, 'column': 29}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (e:Entity)\nRETURN\n  e.entity_id AS entity_id,\n  coalesce(e.embedding_id, e.entity_id) AS embedding_id,\n  coalesce(e.entity_name, e.normalized_value, e.value, '') AS entity_text\n"
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. 

chunks (with embedding_id): 837
entities (with embedding_id): 9359
mentions: 13943
related_to: 7344


(                               chunk_id doc_id  \
 0  019d5909-3849-7db1-8d44-050bee64fe4c     12   
 1  019d5909-384a-77ab-a87c-0c79315c9242     12   
 
                            embedding_id text  
 0  4938f20f-74ed-5f62-b568-289d384cccc8       
 1  6db23d97-c790-5a4d-8969-3f2cb399a48a       ,
                                            entity_id  \
 0  61321ef231397a7420acbcb84cc57cd74a1f520f8a4f94...   
 1  2246cc1e9afc11f58ec447ae8eb7cd7f6091f5b4672428...   
 
                            embedding_id         entity_text  
 0  97b8a0e4-ab00-5bc0-acbf-417d96dd9489    modern anarchism  
 1  7648365a-b3fa-5cc6-8498-6ebc281e99bd  anarchist movement  ,
                                chunk_id  \
 0  019d5909-3849-7db1-8d44-050bee64fe4c   
 1  019d5909-3849-7db1-8d44-050bee64fe4c   
 
                                            entity_id  
 0  27c3f8025dd98b8b23dad1e78503572e31b73ca9c7d50b...  
 1  e712e048610af35a517ffe1e90002d4161a5669e102933...  )

In [34]:
# Загружаем векторы из Qdrant по embedding_id (= point.id в Qdrant)
def _extract_vector(raw_vector):
    if raw_vector is None:
        return None
    if isinstance(raw_vector, dict):
        # named vectors: берём первый
        raw_vector = next(iter(raw_vector.values())) if raw_vector else None
    if raw_vector is None:
        return None
    return np.asarray(raw_vector, dtype=np.float32)


def _normalize_qdrant_point_id(raw_id):
    if raw_id is None or pd.isna(raw_id):
        return None
    if isinstance(raw_id, (int, np.integer)):
        return int(raw_id)

    raw_id = str(raw_id).strip()
    if not raw_id:
        return None
    if raw_id.lstrip('-').isdigit():
        return int(raw_id)
    return raw_id


def fetch_vectors_by_embedding_ids(collection_name: str, embedding_ids: list[str], batch_size: int = 256):
    vectors: dict[str, np.ndarray] = {}
    ids = []
    for raw_id in embedding_ids:
        normalized_id = _normalize_qdrant_point_id(raw_id)
        if normalized_id is not None:
            ids.append(normalized_id)

    ids = list(dict.fromkeys(ids))
    for i in range(0, len(ids), batch_size):
        batch = ids[i:i+batch_size]
        points = qdrant.retrieve(
            collection_name=collection_name,
            ids=batch,
            with_payload=False,
            with_vectors=True,
        )
        for p in points:
            v = _extract_vector(p.vector)
            if v is not None:
                vectors[str(p.id)] = v
    return vectors


chunk_vec_map = fetch_vectors_by_embedding_ids(
    collection_name=QDRANT_CHUNK_COLLECTION,
    embedding_ids=chunks_df['embedding_id'].tolist(),
)
entity_vec_map = fetch_vectors_by_embedding_ids(
    collection_name=QDRANT_ENTITY_COLLECTION,
    embedding_ids=entities_df['embedding_id'].tolist(),
)

chunks_df['vec'] = chunks_df['embedding_id'].astype(str).map(chunk_vec_map)
entities_df['vec'] = entities_df['embedding_id'].astype(str).map(entity_vec_map)

chunks_df = chunks_df[chunks_df['vec'].notna()].copy().reset_index(drop=True)
entities_df = entities_df[entities_df['vec'].notna()].copy().reset_index(drop=True)

#print(entities_df)
print("ENTITIES COUNT: ", entities_df.count())

if chunks_df.empty or entities_df.empty:
    raise ValueError('Не удалось получить вектора из Qdrant. Проверьте collection names и embedding_id mapping.')

chunk_dims = chunks_df['vec'].apply(len).unique().tolist()
entity_dims = entities_df['vec'].apply(len).unique().tolist()
if len(chunk_dims) != 1 or len(entity_dims) != 1:
    raise ValueError(f'Непостоянная размерность векторов: chunk={chunk_dims}, entity={entity_dims}')

chunk_x = np.vstack(chunks_df['vec'].values).astype(np.float32)
entity_x = np.vstack(entities_df['vec'].values).astype(np.float32)

# L2-нормализация входных фичей
chunk_x = chunk_x / (np.linalg.norm(chunk_x, axis=1, keepdims=True) + 1e-12)
entity_x = entity_x / (np.linalg.norm(entity_x, axis=1, keepdims=True) + 1e-12)

chunk2idx = {cid: i for i, cid in enumerate(chunks_df['chunk_id'].tolist())}
entity2idx = {eid: i for i, eid in enumerate(entities_df['entity_id'].tolist())}

mentions_df = mentions_df[mentions_df['chunk_id'].isin(chunk2idx) & mentions_df['entity_id'].isin(entity2idx)].copy()
related_df = related_df[related_df['src_entity_id'].isin(entity2idx) & related_df['dst_entity_id'].isin(entity2idx)].copy()

mention_src = torch.tensor([chunk2idx[x] for x in mentions_df['chunk_id']], dtype=torch.long)
mention_dst = torch.tensor([entity2idx[x] for x in mentions_df['entity_id']], dtype=torch.long)

rel_src = torch.tensor([entity2idx[x] for x in related_df['src_entity_id']], dtype=torch.long) if len(related_df) else torch.empty(0, dtype=torch.long)
rel_dst = torch.tensor([entity2idx[x] for x in related_df['dst_entity_id']], dtype=torch.long) if len(related_df) else torch.empty(0, dtype=torch.long)

data = HeteroData()
data['chunk'].x = torch.from_numpy(chunk_x)
data['entity'].x = torch.from_numpy(entity_x)

data['chunk', 'mentions', 'entity'].edge_index = torch.stack([mention_src, mention_dst], dim=0)
data['entity', 'mentioned_in', 'chunk'].edge_index = torch.stack([mention_dst, mention_src], dim=0)
data['entity', 'related_to', 'entity'].edge_index = torch.stack([rel_src, rel_dst], dim=0)
data['entity', 'related_to_rev', 'entity'].edge_index = torch.stack([rel_dst, rel_src], dim=0)

print(f'Loaded Qdrant vectors: chunks={len(chunks_df)} (dim={chunk_x.shape[1]}), entities={len(entities_df)} (dim={entity_x.shape[1]})')
data


ENTITIES COUNT:  entity_id       9359
embedding_id    9359
entity_text     9359
vec             9359
dtype: int64
Loaded Qdrant vectors: chunks=837 (dim=256), entities=9359 (dim=256)


HeteroData(
  chunk={ x=[837, 256] },
  entity={ x=[9359, 256] },
  (chunk, mentions, entity)={ edge_index=[2, 13943] },
  (entity, mentioned_in, chunk)={ edge_index=[2, 13943] },
  (entity, related_to, entity)={ edge_index=[2, 7344] },
  (entity, related_to_rev, entity)={ edge_index=[2, 7344] }
)

In [35]:
# Train/Val/Test split по рёбрам MENTIONS
all_edges = data['chunk', 'mentions', 'entity'].edge_index.t().cpu().numpy()
perm = np.random.permutation(len(all_edges))
all_edges = all_edges[perm]

n = len(all_edges)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_edges = torch.tensor(all_edges[:n_train], dtype=torch.long)
val_edges = torch.tensor(all_edges[n_train:n_train+n_val], dtype=torch.long)
test_edges = torch.tensor(all_edges[n_train+n_val:], dtype=torch.long)

# Для честной валидации/теста убираем val/test MENTIONS-рёбра из графа обучения.
train_edge_index = train_edges.t().contiguous()
train_data = data.clone()
train_data['chunk', 'mentions', 'entity'].edge_index = train_edge_index
train_data['entity', 'mentioned_in', 'chunk'].edge_index = train_edge_index.flip(0)

num_chunks = train_data['chunk'].x.shape[0]
num_entities = train_data['entity'].x.shape[0]

existing_train = set((int(c), int(e)) for c, e in train_edges.tolist())
positives_by_chunk: dict[int, set[int]] = {}
for c_idx, e_idx in train_edges.tolist():
    positives_by_chunk.setdefault(int(c_idx), set()).add(int(e_idx))


def build_hard_negative_pools(
    chunk_features: np.ndarray,
    entity_features: np.ndarray,
    positives_map: dict[int, set[int]],
    top_k: int = 32,
    batch_size: int = 128,
):
    chunk_tensor = torch.from_numpy(chunk_features)
    entity_tensor = torch.from_numpy(entity_features)
    pools: dict[int, list[int]] = {}

    for start in range(0, len(chunk_tensor), batch_size):
        stop = min(start + batch_size, len(chunk_tensor))
        scores = torch.matmul(chunk_tensor[start:stop], entity_tensor.t())
        candidate_count = min(entity_tensor.shape[0], top_k + 64)
        top_idx = torch.topk(scores, k=candidate_count, dim=1).indices.cpu().tolist()

        for local_idx, candidates in enumerate(top_idx):
            chunk_idx = start + local_idx
            positives = positives_map.get(chunk_idx, set())
            hard_candidates = []
            for entity_idx in candidates:
                entity_idx = int(entity_idx)
                if entity_idx in positives:
                    continue
                hard_candidates.append(entity_idx)
                if len(hard_candidates) == top_k:
                    break
            pools[chunk_idx] = hard_candidates

    return pools


hard_negative_top_k = 32
hard_negative_pools = build_hard_negative_pools(
    chunk_features=chunk_x,
    entity_features=entity_x,
    positives_map=positives_by_chunk,
    top_k=hard_negative_top_k,
)


def sample_negative_entities(
    batch_edges: torch.Tensor,
    num_random_neg: int = 4,
    num_hard_neg: int = 2,
    max_tries: int = 50,
) -> torch.Tensor:
    total_neg = num_random_neg + num_hard_neg
    neg_entity_ids = torch.empty((len(batch_edges), total_neg), dtype=torch.long)

    for i, (c_idx, e_idx) in enumerate(batch_edges.tolist()):
        c_idx = int(c_idx)
        e_idx = int(e_idx)
        positives = positives_by_chunk.get(c_idx, set())
        selected = set()

        for hard_entity in hard_negative_pools.get(c_idx, []):
            if hard_entity == e_idx or hard_entity in positives or hard_entity in selected:
                continue
            selected.add(hard_entity)
            if len(selected) == num_hard_neg:
                break

        while len(selected) < total_neg:
            found = False
            for _ in range(max_tries):
                candidate = random.randrange(num_entities)
                if candidate != e_idx and candidate not in positives and candidate not in selected:
                    selected.add(candidate)
                    found = True
                    break
            if not found:
                break

        if len(selected) < total_neg:
            raise RuntimeError(f'Не удалось набрать {total_neg} negatives для chunk={c_idx}')

        neg_entity_ids[i] = torch.tensor(list(selected), dtype=torch.long)

    return neg_entity_ids

print('Hard-negative pools prepared for chunks:', len(hard_negative_pools))
len(train_edges), len(val_edges), len(test_edges)


Hard-negative pools prepared for chunks: 837


(11154, 1394, 1395)

In [36]:
class RetrievalHeteroSAGE(nn.Module):
    def __init__(self, hidden_dim=256, out_dim=128):
        super().__init__()
        self.conv1 = HeteroConv({
            ('chunk', 'mentions', 'entity'): SAGEConv((-1, -1), hidden_dim),
            ('entity', 'mentioned_in', 'chunk'): SAGEConv((-1, -1), hidden_dim),
            ('entity', 'related_to', 'entity'): SAGEConv((-1, -1), hidden_dim),
            ('entity', 'related_to_rev', 'entity'): SAGEConv((-1, -1), hidden_dim),
        }, aggr='mean')
        self.conv2 = HeteroConv({
            ('chunk', 'mentions', 'entity'): SAGEConv((-1, -1), out_dim),
            ('entity', 'mentioned_in', 'chunk'): SAGEConv((-1, -1), out_dim),
            ('entity', 'related_to', 'entity'): SAGEConv((-1, -1), out_dim),
            ('entity', 'related_to_rev', 'entity'): SAGEConv((-1, -1), out_dim),
        }, aggr='mean')

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {k: F.relu(v) for k, v in x_dict.items()}
        x_dict = self.conv2(x_dict, edge_index_dict)
        x_dict = {k: F.normalize(v, p=2, dim=-1) for k, v in x_dict.items()}
        return x_dict


def edge_scores(chunk_emb: torch.Tensor, entity_emb: torch.Tensor, edges: torch.Tensor) -> torch.Tensor:
    c = chunk_emb[edges[:, 0]]
    e = entity_emb[edges[:, 1]]
    return (c * e).sum(dim=-1)


@torch.no_grad()
def evaluate_embeddings(chunk_emb: torch.Tensor, entity_emb: torch.Tensor, eval_edges: torch.Tensor, k_list=(5, 10, 20)):
    recalls = {k: [] for k in k_list}
    mrr = []

    for c_idx, e_idx in eval_edges.tolist():
        scores = torch.matmul(entity_emb, chunk_emb[c_idx])
        ranking = torch.argsort(scores, descending=True)
        rank_pos = (ranking == e_idx).nonzero(as_tuple=False)
        if len(rank_pos) == 0:
            continue
        rank = int(rank_pos[0].item()) + 1

        for k in k_list:
            recalls[k].append(1.0 if rank <= k else 0.0)
        mrr.append(1.0 / rank)

    metrics = {f'Recall@{k}': float(np.mean(v)) if v else 0.0 for k, v in recalls.items()}
    metrics['MRR'] = float(np.mean(mrr)) if mrr else 0.0
    return metrics


@torch.no_grad()
def evaluate_retrieval(model: nn.Module, graph_data: HeteroData, eval_edges: torch.Tensor, k_list=(5, 10, 20)):
    model.eval()
    z = model(graph_data.x_dict, graph_data.edge_index_dict)
    return evaluate_embeddings(z['chunk'], z['entity'], eval_edges, k_list=k_list)


In [37]:
model = RetrievalHeteroSAGE(hidden_dim=256, out_dim=128).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)

train_data = train_data.to(DEVICE)
train_edges = train_edges.to(DEVICE)
val_edges = val_edges.to(DEVICE)
test_edges = test_edges.to(DEVICE)

num_random_neg = 4
num_hard_neg = 2
max_epochs = 120
patience = 15
best_val_mrr = float('-inf')
best_state = None
patience_left = patience

for epoch in range(1, max_epochs + 1):
    model.train()
    optimizer.zero_grad()

    z = model(train_data.x_dict, train_data.edge_index_dict)
    chunk_batch = z['chunk'][train_edges[:, 0]]
    pos_entity_batch = z['entity'][train_edges[:, 1]]
    pos_scores = (chunk_batch * pos_entity_batch).sum(dim=-1, keepdim=True)

    neg_entity_ids = sample_negative_entities(
        train_edges.cpu(),
        num_random_neg=num_random_neg,
        num_hard_neg=num_hard_neg,
    ).to(DEVICE)
    neg_entity_batch = z['entity'][neg_entity_ids]
    neg_scores = (neg_entity_batch * chunk_batch.unsqueeze(1)).sum(dim=-1)

    logits = torch.cat([pos_scores, neg_scores], dim=1)
    labels = torch.zeros(logits.shape[0], dtype=torch.long, device=DEVICE)
    loss = F.cross_entropy(logits, labels)

    loss.backward()
    optimizer.step()

    val_metrics = evaluate_retrieval(model, train_data, val_edges)
    val_mrr = val_metrics['MRR']
    if val_mrr > best_val_mrr:
        best_val_mrr = val_mrr
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_left = patience
    else:
        patience_left -= 1

    if epoch % 5 == 0 or epoch == 1:
        print(
            f'Epoch {epoch:03d} | loss={loss.item():.4f} | '
            f'negatives=random:{num_random_neg}+hard:{num_hard_neg} | '
            f'val={val_metrics} | patience_left={patience_left}'
        )

    if patience_left == 0:
        print(f'Early stopping at epoch {epoch:03d}; best_val_mrr={best_val_mrr:.6f}')
        break

if best_state is not None:
    model.load_state_dict(best_state)

final_val_metrics = evaluate_retrieval(model, train_data, val_edges)
test_metrics = evaluate_retrieval(model, train_data, test_edges)
print('Best val metrics:', final_val_metrics)
print('Test metrics:', test_metrics)


Epoch 001 | loss=1.9442 | negatives=random:4+hard:2 | val={'Recall@5': 0.002152080344332855, 'Recall@10': 0.003586800573888092, 'Recall@20': 0.006456241032998565, 'MRR': 0.002019390279925151} | patience_left=15
Epoch 005 | loss=1.8697 | negatives=random:4+hard:2 | val={'Recall@5': 0.005738880918220947, 'Recall@10': 0.007173601147776184, 'Recall@20': 0.014347202295552367, 'MRR': 0.0036682683296663642} | patience_left=14
Epoch 010 | loss=1.7817 | negatives=random:4+hard:2 | val={'Recall@5': 0.013629842180774749, 'Recall@10': 0.017934002869440458, 'Recall@20': 0.03156384505021521, 'MRR': 0.010050361360145789} | patience_left=15
Epoch 015 | loss=1.6852 | negatives=random:4+hard:2 | val={'Recall@5': 0.012195121951219513, 'Recall@10': 0.01721664275466284, 'Recall@20': 0.025107604017216643, 'MRR': 0.007687268459921756} | patience_left=12
Epoch 020 | loss=1.6122 | negatives=random:4+hard:2 | val={'Recall@5': 0.01649928263988522, 'Recall@10': 0.027259684361549498, 'Recall@20': 0.043758967001434

In [38]:
# Baseline без GNN: retrieval напрямую по исходным Qdrant-векторам
baseline_chunk_emb = torch.from_numpy(chunk_x).to(DEVICE)
baseline_entity_emb = torch.from_numpy(entity_x).to(DEVICE)

baseline_val_metrics = evaluate_embeddings(baseline_chunk_emb, baseline_entity_emb, val_edges)
baseline_test_metrics = evaluate_embeddings(baseline_chunk_emb, baseline_entity_emb, test_edges)

gnn_val_metrics = final_val_metrics
gnn_test_metrics = test_metrics

print('Baseline val metrics:', baseline_val_metrics)
print('GNN val metrics     :', gnn_val_metrics)
print('Val delta (GNN - baseline):', {
    k: gnn_val_metrics[k] - baseline_val_metrics[k]
    for k in gnn_val_metrics
})
print()
print('Baseline test metrics:', baseline_test_metrics)
print('GNN test metrics     :', gnn_test_metrics)
print('Test delta (GNN - baseline):', {
    k: gnn_test_metrics[k] - baseline_test_metrics[k]
    for k in gnn_test_metrics
})

Baseline val metrics: {'Recall@5': 0.06886657101865136, 'Recall@10': 0.1133428981348637, 'Recall@20': 0.16212338593974174, 'MRR': 0.05094586973003869}
GNN val metrics     : {'Recall@5': 0.027977044476327116, 'Recall@10': 0.04375896700143472, 'Recall@20': 0.06527977044476327, 'MRR': 0.023412966042333443}
Val delta (GNN - baseline): {'Recall@5': -0.040889526542324243, 'Recall@10': -0.06958393113342898, 'Recall@20': -0.09684361549497847, 'MRR': -0.027532903687705244}

Baseline test metrics: {'Recall@5': 0.060215053763440864, 'Recall@10': 0.0960573476702509, 'Recall@20': 0.15913978494623657, 'MRR': 0.045165830396866}
GNN test metrics     : {'Recall@5': 0.02078853046594982, 'Recall@10': 0.032974910394265235, 'Recall@20': 0.06236559139784946, 'MRR': 0.019022214810179617}
Test delta (GNN - baseline): {'Recall@5': -0.039426523297491044, 'Recall@10': -0.06308243727598567, 'Recall@20': -0.09677419354838711, 'MRR': -0.026143615586686385}


In [ ]:
# Инференс эмбеддингов и запись обратно в Neo4j
# После честной оценки на train-графе считаем финальные эмбеддинги на полном графе.
data = data.to(DEVICE)
model.eval()
with torch.no_grad():
    z = model(data.x_dict, data.edge_index_dict)

chunk_emb = z['chunk'].detach().cpu().numpy()
entity_emb = z['entity'].detach().cpu().numpy()

chunk_records = [
    {'chunk_id': cid, 'gnn_embedding': emb.tolist()}
    for cid, emb in zip(chunks_df['chunk_id'].tolist(), chunk_emb)
]
entity_records = [
    {'entity_id': eid, 'gnn_embedding': emb.tolist()}
    for eid, emb in zip(entities_df['entity_id'].tolist(), entity_emb)
]

with driver.session() as session:
    session.run("""
    UNWIND $rows AS row
    MATCH (c:Chunk {chunk_id: row.chunk_id})
    SET c.gnn_embedding = row.gnn_embedding,
        c.gnn_model_version = $model_version,
        c.gnn_updated_at = datetime()
    """, rows=chunk_records, model_version='graphsage_v1')

    session.run("""
    UNWIND $rows AS row
    MATCH (e:Entity {entity_id: row.entity_id})
    SET e.gnn_embedding = row.gnn_embedding,
        e.gnn_model_version = $model_version,
        e.gnn_updated_at = datetime()
    """, rows=entity_records, model_version='graphsage_v1')

print('Embeddings upserted:', len(chunk_records), 'chunks,', len(entity_records), 'entities')


In [ ]:
# Demo retrieval: top-N entities для произвольного chunk
sample_chunk_idx = 0
sample_chunk_id = chunks_df.iloc[sample_chunk_idx]['chunk_id']
sample_text = chunks_df.iloc[sample_chunk_idx]['text'][:220]

data = data.to(DEVICE)
with torch.no_grad():
    z = model(data.x_dict, data.edge_index_dict)
    scores = torch.matmul(z['entity'], z['chunk'][sample_chunk_idx])
    topk = torch.topk(scores, k=min(10, len(scores))).indices.cpu().tolist()

print('Chunk:', sample_chunk_id)
print('Text preview:', sample_text)
print('Top entities:')
for i, idx in enumerate(topk, 1):
    row = entities_df.iloc[idx]
    print(f'{i:02d}. {row.entity_id} | {row.entity_text}')


In [ ]:
driver.close()
